# RAG with an Open-Source LLM: Mistral-7B on the SDF Document

**Document:** `sample-sdf-document.pdf` (Cytiva storage-conditions letter + two Certificates of Quality, 3 pages)
**Stack:** PyMuPDF + LlamaIndex + BGE-small embeddings + **Mistral-7B-Instruct v0.2 (GGUF, run locally with llama.cpp)**
**Author:** Bashar

Same retrieval pipeline as the earlier tasks, but the answer is generated by Mistral-7B running inside this notebook instead of by Gemini. No API key is needed, and no document text leaves the machine.

**Before running:** Runtime → Change runtime type → **T4 GPU**. The notebook also runs on CPU, just much more slowly.

## Step 0: Install dependencies

**Why this differs from the course code:** the course installs `llama-cpp-python==0.2.90` and then `llama-index-llms-llama-cpp`. The current version of that LlamaIndex package requires `llama-cpp-python` 0.3 or newer, so the second install quietly replaces the GPU build with a CPU one compiled from source (slow, and no GPU afterwards). Installing a prebuilt **0.3.x CUDA wheel** first satisfies both, and the last line checks that GPU support survived.

In [ ]:
%pip install -q llama-cpp-python==0.3.19 --only-binary=llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
%pip install -q pymupdf llama-index-core llama-index-llms-llama-cpp llama-index-embeddings-huggingface huggingface_hub

import llama_cpp
print("llama-cpp-python", llama_cpp.__version__, "| GPU offload supported:", llama_cpp.llama_supports_gpu_offload())

## Step 1: Download Mistral-7B (GGUF, 4-bit) and upload the PDF

In [1]:
import os, glob

# Same model file as the course: Mistral-7B-Instruct v0.2, Q4_K_M quantisation (~4.4 GB)
MODEL_PATH = os.environ.get("MISTRAL_GGUF", "/content/mistral.gguf")
if not os.path.exists(MODEL_PATH):
    from huggingface_hub import hf_hub_download
    downloaded = hf_hub_download("TheBloke/Mistral-7B-Instruct-v0.2-GGUF", "mistral-7b-instruct-v0.2.Q4_K_M.gguf",
                                 local_dir=os.path.dirname(MODEL_PATH))
    os.rename(downloaded, MODEL_PATH)
print(f"Model: {MODEL_PATH} ({os.path.getsize(MODEL_PATH) / 1024**3:.2f} GB)")

candidates = glob.glob("*sdf-document*.pdf") or glob.glob("/content/*sdf-document*.pdf")
if not candidates:
    from google.colab import files
    print("Please select 'sample-sdf-document.pdf' to upload:")
    uploaded = files.upload()
    candidates = list(uploaded.keys())
pdf_path = candidates[0]
print("Using PDF:", pdf_path)

Model: models/mistral.gguf (4.07 GB)
Using PDF: sample-sdf-document.pdf


## Step 2: Load Mistral into LlamaIndex

Settings changed from the course code, each for a reason:

| Setting | Course | Here | Why |
|---|---|---|---|
| `n_gpu_layers` | 1 | -1 (all layers) on GPU, 0 on CPU | With 1, only 1 of Mistral's 32 layers runs on the GPU and the rest run on the CPU. |
| `temperature` | 0.7 | 0.1 | Compliance questions need the same factual answer every time, not varied phrasing. |
| `context_window` | 2048 | 4096 | Leaves room for retrieved chunks plus the answer. Mistral supports 32k, so this is well within its limit. |
| Prompt format | not set | Mistral `[INST] ... [/INST]` | Mistral-Instruct was trained on this format. Without it the model sees raw text and is more likely to ramble or continue the prompt. |

In [2]:
import time
import llama_cpp
from llama_index.core import Settings
from llama_index.llms.llama_cpp import LlamaCPP

N_GPU_LAYERS = -1 if llama_cpp.llama_supports_gpu_offload() else 0

def completion_to_prompt(completion, system_prompt=None):
    # Mistral-Instruct format. llama.cpp adds the <s> start token itself.
    system = (system_prompt.strip() + "\n\n") if system_prompt else ""
    return f"[INST] {system}{completion.strip()} [/INST]"

def messages_to_prompt(messages, system_prompt=None):
    # Mistral has no system role, so any system message is folded into the first user turn
    prompt, system = "", system_prompt
    for m in messages:
        role = m.role.value
        if role == "system":
            system = m.content
        elif role == "user":
            prompt += completion_to_prompt(m.content, system)
            system = None
        else:
            prompt += f" {m.content}</s>"
    return prompt

t0 = time.perf_counter()
llm = LlamaCPP(
    model_path=MODEL_PATH,
    temperature=0.1,
    max_new_tokens=384,
    context_window=4096,
    completion_to_prompt=completion_to_prompt,
    messages_to_prompt=messages_to_prompt,
    model_kwargs={"n_gpu_layers": N_GPU_LAYERS, "seed": 42},
    verbose=False,
)
Settings.llm = llm
print(f"Mistral loaded in {time.perf_counter() - t0:.1f}s on {'GPU' if N_GPU_LAYERS else 'CPU'}")

Mistral loaded in 4.2s on CPU


## Step 3: Build the index (same as the previous tasks)

One `Document` per page, 128-token chunks with 20 overlap, BGE-small embeddings with its query prefix. Keeping retrieval identical to the Gemini runs means any difference in the answers comes from the LLM.

`top_k=3` is used to keep Mistral's prompts short. That is enough for the sterilization question (both certificates' sterilization rows rank in the top 3), but, as the retriever-tuning task showed, not enough for the full list of test methods.

In [3]:
import fitz  # PyMuPDF
from llama_index.core import Document, VectorStoreIndex, get_response_synthesizer
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

TOP_K = 3

def load_pdf_with_pymupdf(path):
    # Parse a PDF into one LlamaIndex Document per page.
    doc = fitz.open(path)
    documents = []
    for i, page in enumerate(doc):
        text = page.get_text()
        if not text.strip():
            continue
        documents.append(Document(text=text, metadata={"page_number": i + 1},
                                  excluded_embed_metadata_keys=["page_number"]))
    doc.close()
    return documents

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5",
    query_instruction="Represent this sentence for searching relevant passages: ")

nodes = SentenceSplitter(chunk_size=128, chunk_overlap=20).get_nodes_from_documents(load_pdf_with_pymupdf(pdf_path))
index = VectorStoreIndex(nodes)
retriever = index.as_retriever(similarity_top_k=TOP_K)
synthesizer = get_response_synthesizer(llm=llm, response_mode="compact")
print(f"{len(nodes)} chunks indexed")

12 chunks indexed


## Step 4: Ask Mistral about the document

The course question first, then the three questions used with Gemini in the earlier tasks. For each one: the retrieved chunks, Mistral's answer, and how long generation took.

In [4]:
QUERIES = [
    "What sterilization method was used for this product?",
    "What are the storage conditions for this product?",
    "What test methods were used for quality control?",
    "Which bio-safety and regulatory standards do the materials comply with?",
]

results = []
for query in QUERIES:
    retrieved = retriever.retrieve(query)

    t0 = time.perf_counter()
    answer = str(synthesizer.synthesize(query, nodes=retrieved)).strip() or "(empty response)"
    gen_s = time.perf_counter() - t0
    n_tokens = len(llm._model.tokenize(answer.encode("utf-8"), add_bos=False))

    results.append({"query": query, "answer": answer, "seconds": gen_s, "tokens": n_tokens,
                    "pages": [n.metadata["page_number"] for n in retrieved]})

    print(f"\n{'=' * 100}\nQ: {query}\n{'=' * 100}")
    for r, n in enumerate(retrieved, 1):
        print(f"  [{r}] score={n.score:.3f} page {n.metadata['page_number']} | " + n.get_content()[:110].replace("\n", " | "))
    print(f"\nMISTRAL ({gen_s:.1f}s, {n_tokens} tokens, {n_tokens / gen_s:.1f} tokens/s):\n{answer}")


Q: What sterilization method was used for this product?
  [1] score=0.582 page 2 | Description  | Limits  | Autoclave | Pump tubing | 121°C > 15 min  | Gamma Irradiation  | Inlets  | 25.0 – 40.0 kGy  | Regulat
  [2] score=0.551 page 2 | cytiva.com  | 29190939 Rev AD | Product: ÄKTA™ ready Gradient Flow Section With Inlets | Lot Number: 17242818  | Produ
  [3] score=0.530 page 3 | Product: | AKTA ready Low Flow Kit | Lot Number: | 18356721 | Product Article Number: | 28 9301 82 | Date of Manufacture: | 

MISTRAL (23.2s, 97 tokens, 4.2 tokens/s):
Based on the context information provided, the product has undergone autoclave sterilization for the pump tubing at a temperature of 121°C for more than 15 minutes, and gamma irradiation for the inlets with a dose range of 25.0 – 40.0 kGy. Therefore, both autoclave and gamma irradiation sterilization methods have been used for this product.



Q: What are the storage conditions for this product?
  [1] score=0.661 page 1 | Cytiva | 100 Results Way | Marlborough, MA 01752 | United States | Page 1 / 1 | cytiva.com | 3 June, 2022 | Re: ÄKTATM ready
  [2] score=0.613 page 1 | Extended storage below the recommended +5 | could lead to  | brittleness or cracking of the plastic connectors. Ho
  [3] score=0.590 page 2 | Description  | Limits  | Autoclave | Pump tubing | 121°C > 15 min  | Gamma Irradiation  | Inlets  | 25.0 – 40.0 kGy  | Regulat

MISTRAL (26.3s, 146 tokens, 5.5 tokens/s):
The recommended storage temperature for ÄKTA ready flow kits is above +5°C, as stated in the Operating Instructions 28960345. However, it is important to note that the operating temperature for these kits is between +2°C and +40°C. Allowing the kits to acclimate to a warmer temperature before use can help reduce the risk of damage during setup and handling. The kits should also comply with USP <88> Biological Reactivity Test Class VI and be free from


Q: What test methods were used for quality control?
  [1] score=0.580 page 3 | 0 - 40.0 kGy | Conforms | Flow Rate Test | Per SOP-QC-042 | Pass | Pressure Integrity | Max 0.5 bar, 5 min hold | Pass | Visua
  [2] score=0.534 page 3 | Cytiva | cytiva.com | Certificate of Quality | This product is manufactured in compliance with our ISO 9001 certifie
  [3] score=0.527 page 2 | Modified, ÄKTATM ready | Expiration Date:  20230126  | Product Release Criteria  | We hereby certify that the define

MISTRAL (32.2s, 200 tokens, 6.2 tokens/s):
Based on the context information provided, the following test methods were used for quality control:

1. Flow Rate Test: This test was performed in accordance with SOP-QC-042.
2. Pressure Integrity Test: The maximum pressure was held at 0.5 bar for a duration of 5 minutes.
3. Visual Inspection: No defects were visible during this inspection.
4. Package Integrity Test: The package was found to be sealed and undamaged.
5. Regulatory Conformance Test: The poly


Q: Which bio-safety and regulatory standards do the materials comply with?
  [1] score=0.733 page 2 | Description  | Limits  | Autoclave | Pump tubing | 121°C > 15 min  | Gamma Irradiation  | Inlets  | 25.0 – 40.0 kGy  | Regulat
  [2] score=0.718 page 3 | 0 - 40.0 kGy | Conforms | Flow Rate Test | Per SOP-QC-042 | Pass | Pressure Integrity | Max 0.5 bar, 5 min hold | Pass | Visua
  [3] score=0.625 page 2 | Certificate of Quality  | This product is manufactured in compliance with our ISO 9001 certified quality managem

MISTRAL (20.6s, 80 tokens, 3.9 tokens/s):
The materials in contact with the process fluid comply with the United States Pharmacopeia (USP) <88> Biological Reactivity Test Class VI standards for bio-safety. Additionally, they are free from animal derived ingredients or in compliance with EMA/410/01, and conform to regulatory requirements. The country of origin for the materials is USA.


## Step 5: Summary

In [5]:
import pandas as pd

summary = pd.DataFrame([{
    "Query": r["query"],
    "Pages retrieved": r["pages"],
    "Answer tokens": r["tokens"],
    "Generation (s)": round(r["seconds"], 1),
    "Tokens/s": round(r["tokens"] / r["seconds"], 1),
} for r in results])
summary

,Query,Pages retrieved,Answer tokens,Generation (s),Tokens/s
0,What sterilization method was used for this pr...,"[2, 2, 3]",97,23.2,4.2
1,What are the storage conditions for this product?,"[1, 1, 2]",146,26.3,5.5
2,What test methods were used for quality control?,"[3, 3, 2]",200,32.2,6.2
3,Which bio-safety and regulatory standards do t...,"[2, 3, 2]",80,20.6,3.9


## Findings

Results below are from a run on a laptop CPU (no GPU), with the same model file and settings as the Colab version. On a Colab T4 GPU, generation is much faster; the answers should be very close but are not guaranteed to be word-for-word identical.

**1. Mistral answered every question correctly from the retrieved text.** The course question, "What sterilization method was used?", got a complete and accurate answer: autoclave for the pump tubing (121°C, more than 15 min) and gamma irradiation for the inlets (25.0 to 40.0 kGy). It did not invent any values. Every number in its answers can be found in the retrieved chunks.

**2. It pads answers with nearby but irrelevant facts.** This is the clearest weakness. Asked about **storage conditions**, it gave the right answer (above +5°C, operating range +2°C to +40°C) and then added that the kits "should also comply with USP <88>... and EMA/410/01" and that the country of origin is the USA. Those facts come from the third retrieved chunk, which is a sterilization and bio-safety table from a different page, and they have nothing to do with storage. The test-methods and bio-safety answers have the same habit (ISO 9001, "conform to regulatory requirements", country of origin). Mistral tends to use everything it is given rather than only what answers the question.

**3. It also leaves things out.** In the storage answer, Mistral dropped the warning that storage below +5°C can make the plastic connectors brittle or crack, even though that sentence was in its second chunk. Gemini included it in the embedding comparison.

**4. Retrieval limits carry over unchanged.** For test methods, Mistral listed Flow Rate, Pressure Integrity, Visual Inspection and Package Integrity, the same four of six that Gemini gave at a low `top_k`, because the Autoclave and Gamma Irradiation rows were not in the retrieved chunks. Swapping the LLM cannot fix what retrieval did not find.

**5. Speed on CPU is slow but usable for testing.** About 4 to 6 tokens per second, so roughly 20 to 32 seconds per answer for 80 to 200 tokens. Answers that were padded took longest. The model loaded in about 5 seconds once downloaded; the 4.4 GB download is the main one-off cost.

**6. Practical points that the course code gets wrong.**
- **Install order:** installing `llama-index-llms-llama-cpp` after `llama-cpp-python==0.2.90` replaces the GPU build, because the LlamaIndex package now needs version 0.3 or newer. This notebook installs a prebuilt 0.3.19 CUDA wheel and checks that GPU support is still on.
- **GPU layers:** `n_gpu_layers=1` puts only 1 of 32 layers on the GPU. `-1` puts the whole model there.
- **Prompt format:** Mistral-Instruct expects `[INST] ... [/INST]`, which LlamaIndex does not apply by default for a local model.

**Takeaway for the Gemini comparison.** On this document Mistral is accurate but less disciplined than Gemini: longer answers, extra off-topic details, and the occasional omission. In exchange, it runs with no API key and no rate limit, and the document never leaves the machine. That matters here, given that the free Gemini tier ran out after 20 requests during the earlier tasks.